### Imports

In [ ]:
import zipfile
from pathlib import Path

import pandas as pd
from python_calamine import load_workbook

### Read the data from the Excel file

In [ ]:
RAW_PATH = Path("../data/raw")
assert RAW_PATH.exists()

In [ ]:
SIMULADOR_FILE = RAW_PATH / "Simulador Bloco e Subbloco_v2.xlsx"
assert SIMULADOR_FILE.exists()

In [ ]:
excel = pd.ExcelFile(SIMULADOR_FILE, engine="calamine")
print(excel.sheet_names)

# Data cleaning steps

1. Extract 'Calendário FV' sheet from the provided Excel file.
    - extract sheet by sheet name
    - drop empty columns
2. Extract 'Base Demanda' sheet from the provided Excel file.
    - extract sheet by sheet name
    - filter columns (not calculated columns)

### Extract calendar data from the 'Calendário FV' sheet

In [ ]:
df_calendario = excel.parse("Calendário FV")

In [ ]:
# drop empty columns
df_calendario = df_calendario.dropna(how="all", axis=1)

In [ ]:
df_calendario.head()

In [ ]:
# export csv
output_path = RAW_PATH / "calendario_fv.csv"
df_calendario.to_csv(output_path, index=False)

### Extract demand from  'Base Demanda' sheet

Notes:
- Dia Captacao depends on data from calendar that seems to not be present in the 'Calendário FV' sheet. It is possible that this data is in another sheet or file. Further investigation is needed to locate the source of this information. Or even if this is needed. Initial thought it that it's not needed.

In [ ]:
df_demand = excel.parse("Base Demanda")

In [ ]:
DEMAND_COLUMNS_NEEDED = [
    "data_pedido",
    "cd_setor",
    "cd_cd",
    "nm_ciclo",
    "aa_ciclo",
    "total_pedidos",
    "total_volumes",
    "total_itens",
]
df_demand = df_demand[DEMAND_COLUMNS_NEEDED]

In [ ]:
df_demand.head()

In [ ]:
row = df_demand.iloc[40]
row

In [ ]:
calendario_per_sector = df_calendario[df_calendario["COD SETOR"] == row["cd_setor"]]

In [ ]:
calendario_per_sector[
    (calendario_per_sector["Dt Abertura"] <= row["data_pedido"])
    & (calendario_per_sector["Dt Fechamento"] >= row["data_pedido"])
]

In [ ]:
# I expect that every cycle start and end at the same day if the sector is in
# the same block + sublock
# this is the code to check this hypothesis is correct
is_consistent = (
    df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[["Dt Abertura", "Dt Fechamento"]]
    .nunique()
    .eq(1)
    .all()
    .all()
)

print(f"Expectation holds: {is_consistent}")

In [ ]:
# which blocks/subblocks/cycles diverge (and which sectors differ):
# Count unique start and end dates per block + subblock + cycle
cycle_dates_summary = df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[
    ["Dt Abertura", "Dt Fechamento"]
].nunique()

# Filter for groups that have more than 1 distinct date
discrepancies = cycle_dates_summary[
    (cycle_dates_summary["Dt Abertura"] > 1) | (cycle_dates_summary["Dt Fechamento"] > 1)
]

if discrepancies.empty:
    print(
        "Confirmed: Every cycle starts and ends on the same day for "
        "all sectors in the same block + sublock."
    )
else:
    print(
        f"Found {len(discrepancies)} (BLOCO, SUB BLOCO, CICLOS) combination(s) "
        "with diverging dates:\n"
    )
    display(discrepancies)

    # Inspect the exact sectors and dates causing the mismatch
    conflicting_rows = df_calendario.merge(
        discrepancies.reset_index()[["BLOCO", "SUB BLOCO", "CICLOS"]],
        on=["BLOCO", "SUB BLOCO", "CICLOS"],
    )
    display(
        conflicting_rows[
            ["BLOCO", "SUB BLOCO", "CICLOS", "COD SETOR", "Dt Abertura", "Dt Fechamento"]
        ].sort_values(by=["BLOCO", "SUB BLOCO", "CICLOS", "COD SETOR"])
    )

In [ ]:
# export csv
output_file = RAW_PATH / "demanda.csv"
df_demand.to_csv(output_file, index=False)

In [ ]:
# 1. Quick check for external workbook links in XLSX package relationships
with zipfile.ZipFile(SIMULADOR_FILE) as z:
    ext_links = [name for name in z.namelist() if "externalLink" in name]
    if ext_links:
        print(f"External workbook dependencies found: {ext_links}")
    else:
        print("No external workbook dependencies found in package metadata.")

In [ ]:
# 2. Fast scan for evaluated errors (#REF!, #VALUE!, #DIV/0!, #N/A, etc.) using Calamine
wb = load_workbook(SIMULADOR_FILE)
excel_error_tokens = {"#REF!", "#VALUE!", "#DIV/0!", "#NAME?", "#N/A", "#NUM!", "#NULL!"}
broken_items = []

In [ ]:
def col_to_letter(col_idx: int) -> str:
    """Convert a 0-indexed column integer to Excel column coordinate (e.g. 0 -> A, 27 -> AB)."""
    result = ""
    col_idx += 1
    while col_idx > 0:
        col_idx, remainder = divmod(col_idx - 1, 26)
        result = chr(65 + remainder) + result
    return result


for sheet_name in wb.sheet_names:
    ws = wb.get_sheet_by_name(sheet_name)
    for r_idx, row in enumerate(ws.to_python()):
        for c_idx, val in enumerate(row):
            if isinstance(val, str) and val in excel_error_tokens:
                coord = f"{col_to_letter(c_idx)}{r_idx + 1}"
                broken_items.append(
                    {
                        "sheet": sheet_name,
                        "cell": coord,
                        "issue_type": f"Cached error ({val})",
                        "error": val,
                    }
                )

In [ ]:
df_broken = pd.DataFrame(broken_items)

print(f"Total issues found: {len(df_broken)}")
if not df_broken.empty:
    print("\nSummary by sheet and issue type:")
    print(df_broken.groupby(["sheet", "issue_type"]).size().rename("count"))

df_broken.head(20)